# 2.5 归一化、残差与 FFN / Normalization, residuals and the FFN

让几十层网络训得动的三样东西。最后把所有部件拼起来，完整复现 GPT-2 的前向计算，并训练一个 mini-GPT。

The three things that make dozens of layers trainable. At the end all components are assembled to reproduce GPT-2's forward pass in full and to train a mini-GPT.

讲解与推导见网站对应模块。每个练习先写代码，再运行紧随其后的检查单元。

The explanation and derivations are on the website. For each exercise, write the code, then run the check cell right after it.

## 1. 手写 LayerNorm 与 RMSNorm / LayerNorm and RMSNorm by hand

实现两种归一化，与 `nn.LayerNorm` 和 `nn.RMSNorm` 对比。测试还会验证三条性质，即 LayerNorm 的输出均值为 0、两者都对输入的整体缩放不敏感、输入均值为 0 时两者结果相同。

Implement both normalizations and compare with `nn.LayerNorm` and `nn.RMSNorm`. The tests also verify three properties, namely that the output of LayerNorm has mean 0, that both are insensitive to an overall rescaling of the input, and that the two coincide when the input has mean 0.

In [ ]:
import torch


def layer_norm(x, weight, bias, eps=1e-5):
    """LayerNorm over the last dimension. x: (..., d), weight and bias: (d,).

    Subtract the mean, divide by the standard deviation (biased variance, eps inside
    the square root), then scale by weight and shift by bias.
    Do not call F.layer_norm or nn.LayerNorm.
    """
    # TODO 1
    raise NotImplementedError


def rms_norm(x, weight, eps=1e-6):
    """RMSNorm over the last dimension: divide by sqrt(mean(x^2) + eps), then scale by weight.

    No mean subtraction and no bias.
    """
    # TODO 2
    raise NotImplementedError

In [ ]:
# 检查 / checks
import torch
import torch.nn as nn

torch.manual_seed(0)
d = 32
x = torch.randn(4, 7, d, dtype=torch.float64) * 3 + 2

ln = nn.LayerNorm(d, dtype=torch.float64)
with torch.no_grad():
    ln.weight.uniform_(0.5, 1.5); ln.bias.uniform_(-1, 1)
assert torch.allclose(layer_norm(x, ln.weight, ln.bias), ln(x), atol=1e-10), "differs from nn.LayerNorm"

rn = nn.RMSNorm(d, eps=1e-6, dtype=torch.float64)
with torch.no_grad():
    rn.weight.uniform_(0.5, 1.5)
assert torch.allclose(rms_norm(x, rn.weight), rn(x), atol=1e-10), "differs from nn.RMSNorm"

ones, zeros = torch.ones(d, dtype=torch.float64), torch.zeros(d, dtype=torch.float64)
y = layer_norm(x, ones, zeros)
assert torch.allclose(y.mean(-1), torch.zeros(4, 7, dtype=torch.float64), atol=1e-9), "LayerNorm output has mean 0"
assert torch.allclose(y.pow(2).mean(-1), torch.ones(4, 7, dtype=torch.float64), atol=1e-4), "and variance 1"
z = rms_norm(x, ones)
assert torch.allclose(z.pow(2).mean(-1), torch.ones(4, 7, dtype=torch.float64), atol=1e-4), "RMSNorm output has mean square 1"
assert z.mean(-1).abs().max() > 0.1, "RMSNorm does not remove the mean"

# Both are invariant to rescaling the input, which is what keeps activations from blowing up.
assert torch.allclose(layer_norm(10 * x, ones, zeros), y, atol=1e-5)
assert torch.allclose(rms_norm(10 * x, ones), z, atol=1e-5)
# For an input that already has mean 0, the two coincide.
x0 = x - x.mean(-1, keepdim=True)
assert torch.allclose(layer_norm(x0, ones, zeros, eps=1e-6), rms_norm(x0, ones, eps=1e-6), atol=1e-10)
print("m25_norms: all tests passed")

<details><summary><b>提示 1 / Hint 1</b></summary>

均值和方差都沿最后一维计算，并保留维度，即 `x.mean(dim=-1, keepdim=True)`。

Compute mean and variance along the last dimension and keep it, `x.mean(dim=-1, keepdim=True)`.

</details>

<details><summary><b>提示 2 / Hint 2</b></summary>

方差用除以 $d$ 的版本，即 `((x - mean) ** 2).mean(...)`。不要用 `x.var()`，它默认除以 $d - 1$。`eps` 加在根号里面。

Use the variance that divides by $d$, `((x - mean) ** 2).mean(...)`. Do not use `x.var()`, which divides by $d - 1$ by default. `eps` goes inside the square root.

</details>

<details><summary><b>参考答案 / Reference solution</b></summary>

```python
import torch


def layer_norm(x, weight, bias, eps=1e-5):
    mean = x.mean(dim=-1, keepdim=True)
    var = ((x - mean) ** 2).mean(dim=-1, keepdim=True)       # biased: divide by d, not d - 1
    return (x - mean) / torch.sqrt(var + eps) * weight + bias


def rms_norm(x, weight, eps=1e-6):
    rms = torch.sqrt((x ** 2).mean(dim=-1, keepdim=True) + eps)
    return x / rms * weight
```

</details>

## 2. 复现 GPT-2 的 MLP 并实现 SwiGLU / Reproduce GPT-2's MLP and implement SwiGLU

用权重复现 GPT-2 的 MLP 层。再实现 SwiGLU，并算出让它与标准 FFN 参数量相同的隐藏宽度。

Reproduce the MLP layer of GPT-2 from its weights. Then implement SwiGLU and compute the hidden width at which it has the same number of parameters as the standard FFN.

需要联网下载模型。/ Downloads a model.

In [ ]:
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM

# GPT-2's MLP is  mlp = model.transformer.h[i].mlp  with two Conv1D layers
#   mlp.c_fc    768 -> 3072     (y = x @ weight + bias)
#   mlp.c_proj  3072 -> 768
# and the activation F.gelu(..., approximate="tanh") in between.


@torch.no_grad()
def gpt2_mlp(x, mlp):
    """Recompute mlp(x) from its weights. x: (..., 768)."""
    # TODO 1: widen, apply GELU, narrow
    raise NotImplementedError


def swiglu_ffn(x, w_gate, w_up, w_down):
    """SwiGLU feed-forward layer (no biases), as used in LLaMA-style models.

    x: (..., d)   w_gate, w_up: (d, h)   w_down: (h, d)
    out = ( silu(x @ w_gate) * (x @ w_up) ) @ w_down,   silu(z) = z * sigmoid(z)
    Do not call F.silu.
    """
    # TODO 2
    raise NotImplementedError


def ffn_param_count(d, hidden, kind):
    """Weights only, no biases. kind is "mlp" (two matrices) or "swiglu" (three matrices)."""
    # TODO 3
    raise NotImplementedError


def swiglu_hidden_for_same_params(d):
    """The SwiGLU hidden size that matches the parameter count of a standard MLP with hidden size 4d."""
    # TODO 4: solve  3 * d * h = 2 * d * 4d  for h and return an int (round down)
    raise NotImplementedError

In [ ]:
# 检查 / checks
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM

gpt2 = AutoModelForCausalLM.from_pretrained("openai-community/gpt2").eval()
torch.manual_seed(0)
x = torch.randn(2, 5, 768)
for layer in (0, 11):
    mlp = gpt2.transformer.h[layer].mlp
    with torch.no_grad():
        ref = mlp(x)
    assert torch.allclose(gpt2_mlp(x, mlp), ref, atol=1e-3), f"layer {layer}: differs from GPT-2's MLP"

d, h = 16, 40
xs = torch.randn(3, d, dtype=torch.float64)
wg, wu = torch.randn(d, h, dtype=torch.float64), torch.randn(d, h, dtype=torch.float64)
wd = torch.randn(h, d, dtype=torch.float64)
_silu = F.silu
F.silu = lambda *a, **k: (_ for _ in ()).throw(AssertionError("write SiLU yourself"))
try:
    out = swiglu_ffn(xs, wg, wu, wd)
finally:
    F.silu = _silu
assert out.shape == (3, d)
assert torch.allclose(out, (F.silu(xs @ wg) * (xs @ wu)) @ wd, atol=1e-10), "SwiGLU formula is wrong"

assert ffn_param_count(768, 3072, "mlp") == 2 * 768 * 3072
assert ffn_param_count(768, 2048, "swiglu") == 3 * 768 * 2048
assert swiglu_hidden_for_same_params(768) == 2048, "8d/3: this is where the factor 2/3 in LLaMA-style configs comes from"
assert ffn_param_count(768, swiglu_hidden_for_same_params(768), "swiglu") == ffn_param_count(768, 4 * 768, "mlp")
print("m25_ffn: all tests passed")

<details><summary><b>提示 1 / Hint 1</b></summary>

GPT-2 的 GELU 用的是 tanh 近似，需要写成 `F.gelu(z, approximate="tanh")`，否则数值会有微小差别。

GPT-2 uses the tanh approximation of GELU, written `F.gelu(z, approximate="tanh")`. Without it the numbers differ slightly.

</details>

<details><summary><b>提示 2 / Hint 2</b></summary>

SiLU 是 `g * torch.sigmoid(g)`。

SiLU is `g * torch.sigmoid(g)`.

</details>

<details><summary><b>提示 3 / Hint 3</b></summary>

由 $3dh = 8d^2$ 得 $h = 8d/3$，用整数除法 `8 * d // 3`。

From $3dh = 8d^2$ follows $h = 8d/3$. Use integer division, `8 * d // 3`.

</details>

<details><summary><b>参考答案 / Reference solution</b></summary>

```python
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM


@torch.no_grad()
def gpt2_mlp(x, mlp):
    h = F.gelu(x @ mlp.c_fc.weight + mlp.c_fc.bias, approximate="tanh")     # 768 -> 3072
    return h @ mlp.c_proj.weight + mlp.c_proj.bias                          # 3072 -> 768


def swiglu_ffn(x, w_gate, w_up, w_down):
    g = x @ w_gate
    gate = g * torch.sigmoid(g)                 # SiLU, also called Swish
    return (gate * (x @ w_up)) @ w_down


def ffn_param_count(d, hidden, kind):
    return (2 if kind == "mlp" else 3) * d * hidden


def swiglu_hidden_for_same_params(d):
    return 8 * d // 3
```

</details>

## 3. 拼出完整的 GPT-2 前向计算 / Assemble the full GPT-2 forward pass

用 2.2 写的 `gpt2_attention` 和上一题的 `gpt2_mlp` 拼出一个 Pre-Norm block，再接上嵌入、12 层 block、最后的归一化和 unembedding，得到完整的 logits。全程不调用 GPT-2 任何一层的前向函数。测试会与模型自身的输出对比。

Use `gpt2_attention` from 2.2 and `gpt2_mlp` from the previous exercise to build a Pre-Norm block, then add the embeddings, the 12 blocks, the final normalization and the unembedding to obtain the full logits. No forward function of any GPT-2 layer is called. The tests compare with the model's own output.

需要联网下载模型。/ Downloads a model.

In [ ]:
# 来自练习 m22_gpt2attn 的参考实现，本题会用到 / reference code from exercise m22_gpt2attn, used below
import math
import torch
from transformers import AutoModelForCausalLM


@torch.no_grad()
def gpt2_attention(x, attn, n_head=12):
    B, T, d = x.shape
    q, k, v = (x @ attn.c_attn.weight + attn.c_attn.bias).split(d, dim=-1)
    split = lambda t: t.view(B, T, n_head, d // n_head).transpose(1, 2)        # (B, h, T, 64)
    q, k, v = split(q), split(k), split(v)
    scores = q @ k.transpose(-2, -1) / math.sqrt(d // n_head)
    causal = torch.tril(torch.ones(T, T, dtype=torch.bool))
    scores = scores.masked_fill(~causal, float("-inf"))
    out = torch.softmax(scores, dim=-1) @ v
    out = out.transpose(1, 2).contiguous().view(B, T, d)
    return out @ attn.c_proj.weight + attn.c_proj.bias

In [ ]:
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer

# This exercise reuses gpt2_attention (module 2.2) and gpt2_mlp (previous exercise).
# A GPT-2 block is  block = model.transformer.h[i]  with
#   block.ln_1, block.attn, block.ln_2, block.mlp
# GPT-2 is a Pre-Norm model.


@torch.no_grad()
def gpt2_block(x, block):
    """One full Transformer block, from your own pieces. x: (B, T, 768)."""
    # TODO 1: x = x + attention(LayerNorm_1(x))     use F.layer_norm(x, (768,), ln.weight, ln.bias, ln.eps)
    # TODO 2: x = x + mlp(LayerNorm_2(x))
    raise NotImplementedError


@torch.no_grad()
def gpt2_logits(model, input_ids):
    """The complete GPT-2 forward pass without calling any GPT-2 layer's forward method.

    input_ids: (1, T).  Returns logits (1, T, 50257).
    Token embedding table: model.transformer.wte.weight   (50257, 768)
    Position embedding table: model.transformer.wpe.weight (1024, 768)
    The unembedding shares the token embedding table.
    """
    t = model.transformer
    # TODO 3: embeddings -> 12 x gpt2_block -> final LayerNorm (t.ln_f) -> x @ wte.weight.T
    raise NotImplementedError

In [ ]:
# 检查 / checks
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

tok = AutoTokenizer.from_pretrained("openai-community/gpt2")
gpt2 = AutoModelForCausalLM.from_pretrained("openai-community/gpt2").eval()

torch.manual_seed(0)
x = torch.randn(1, 6, 768)
for layer in (0, 7):
    block = gpt2.transformer.h[layer]
    with torch.no_grad():
        ref = block(x)
        ref = ref[0] if isinstance(ref, tuple) else ref
    assert torch.allclose(gpt2_block(x, block), ref, atol=1e-3), f"block {layer} differs. Is the LayerNorm applied before each sublayer?"

ids = tok("The capital of France is", return_tensors="pt").input_ids
with torch.no_grad():
    ref = gpt2(ids).logits
out = gpt2_logits(gpt2, ids)
assert out.shape == ref.shape
assert torch.allclose(out, ref, atol=5e-3), f"logits differ by up to {(out - ref).abs().max():.4f}"
assert out[0, -1].argmax() == ref[0, -1].argmax()
print("m25_block: all tests passed. Every step of GPT-2's forward pass is now your own code.")
print("  next token:", repr(tok.decode(out[0, -1].argmax())))

<details><summary><b>提示 1 / Hint 1</b></summary>

归一化写成一个小函数 `F.layer_norm(x, (768,), ln.weight, ln.bias, ln.eps)`，block 里用两次，最后的 `ln_f` 用一次。

Write the normalization as a small helper, `F.layer_norm(x, (768,), ln.weight, ln.bias, ln.eps)`. It is used twice in the block and once for the final `ln_f`.

</details>

<details><summary><b>提示 2 / Hint 2</b></summary>

注意顺序是先归一化再进子层，即 `x = x + gpt2_attention(ln(x), block.attn)`。写成 `ln(x + ...)` 就成了 Post-Norm，结果对不上。

Mind the order, normalization first and then the sublayer, `x = x + gpt2_attention(ln(x), block.attn)`. Writing `ln(x + ...)` would be Post-Norm and the result would not match.

</details>

<details><summary><b>提示 3 / Hint 3</b></summary>

嵌入用 `t.wte.weight[input_ids] + t.wpe.weight[torch.arange(T)]`，最后的 unembedding 是 `x @ t.wte.weight.T`。

The embeddings are `t.wte.weight[input_ids] + t.wpe.weight[torch.arange(T)]` and the final unembedding is `x @ t.wte.weight.T`.

</details>

<details><summary><b>参考答案 / Reference solution</b></summary>

```python
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer


def _ln(x, ln):
    return F.layer_norm(x, (x.shape[-1],), ln.weight, ln.bias, ln.eps)


@torch.no_grad()
def gpt2_block(x, block):
    x = x + gpt2_attention(_ln(x, block.ln_1), block.attn)     # read from other positions, add to the stream
    x = x + gpt2_mlp(_ln(x, block.ln_2), block.mlp)            # process each position, add to the stream
    return x


@torch.no_grad()
def gpt2_logits(model, input_ids):
    t = model.transformer
    T = input_ids.shape[1]
    x = t.wte.weight[input_ids] + t.wpe.weight[torch.arange(T)]
    for block in t.h:
        x = gpt2_block(x, block)
    return _ln(x, t.ln_f) @ t.wte.weight.T
```

</details>

## 4. 从零训练一个字符级 mini-GPT / Train a character-level mini-GPT from scratch

用 2.2 写的 `MultiHeadAttention` 补全 `Block` 和 `MiniGPT` 的前向计算，在 Python 自带的一小段文本上训练 300 步，CPU 上只需几秒。测试会检查初始损失接近 $\ln V$、后面的 token 不影响前面的输出、训练后损失明显下降，并打印一段模型生成的文本。

Complete the forward passes of `Block` and `MiniGPT` with the `MultiHeadAttention` from 2.2 and train for 300 steps on a short text that ships with Python, which takes seconds on a CPU. The tests check that the initial loss is near $\ln V$, that later tokens do not affect earlier outputs and that the loss falls clearly after training, and they print a sample of generated text.

In [ ]:
# 来自练习 m22_mha 的参考实现，本题会用到 / reference code from exercise m22_mha, used below
import math
import torch
import torch.nn as nn


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_head):
        super().__init__()
        assert d_model % n_head == 0
        self.n_head, self.d_head = n_head, d_model // n_head
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.proj = nn.Linear(d_model, d_model)

    def forward(self, x, mask=None):
        B, T, d = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)                                   # each (B, T, d)
        split = lambda t: t.view(B, T, self.n_head, self.d_head).transpose(1, 2)  # (B, h, T, d_head)
        q, k, v = split(q), split(k), split(v)
        scores = q @ k.transpose(-2, -1) / math.sqrt(self.d_head)                # (B, h, T, T)
        if mask is not None:
            scores = scores.masked_fill(~mask, float("-inf"))
        weights = torch.softmax(scores, dim=-1)
        out = weights @ v                                                        # (B, h, T, d_head)
        out = out.transpose(1, 2).contiguous().view(B, T, d)                     # glue the heads back together
        return self.proj(out), weights


def mha_param_count(d_model, n_head):
    return (3 * d_model * d_model + 3 * d_model) + (d_model * d_model + d_model)  # independent of n_head

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

# This exercise uses your MultiHeadAttention class from module 2.2.


class Block(nn.Module):
    """A Pre-Norm Transformer block."""

    def __init__(self, d_model, n_head):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadAttention(d_model, n_head)
        self.ln2 = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(nn.Linear(d_model, 4 * d_model), nn.GELU(), nn.Linear(4 * d_model, d_model))

    def forward(self, x, mask):
        # TODO 1: x = x + attention(ln1(x)) with the causal mask   (self.attn returns (output, weights))
        # TODO 2: x = x + mlp(ln2(x))
        raise NotImplementedError


class MiniGPT(nn.Module):
    def __init__(self, vocab_size, block_size=32, d_model=64, n_head=4, n_layer=2):
        super().__init__()
        self.block_size = block_size
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(block_size, d_model)
        self.blocks = nn.ModuleList([Block(d_model, n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, idx, targets=None):
        """idx: (B, T) token ids. Returns (logits (B, T, V), loss or None)."""
        B, T = idx.shape
        # TODO 3: token embedding + position embedding
        # TODO 4: build the causal mask (T, T), True = may attend, and run the blocks
        # TODO 5: final LayerNorm, then the head -> logits
        # TODO 6: if targets is given, loss = cross-entropy between logits (B*T, V) and targets (B*T,)
        raise NotImplementedError

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, generator=None):
        for _ in range(max_new_tokens):
            logits, _ = self(idx[:, -self.block_size:])
            probs = torch.softmax(logits[:, -1], dim=-1)
            idx = torch.cat([idx, torch.multinomial(probs, 1, generator=generator)], dim=1)
        return idx


def train(model, data, steps=300, batch_size=32, lr=3e-3, seed=0):
    """Train on a 1-D tensor of token ids. Returns the list of loss values. Nothing to change here."""
    g = torch.Generator().manual_seed(seed)
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    losses = []
    for _ in range(steps):
        ix = torch.randint(len(data) - model.block_size - 1, (batch_size,), generator=g)
        x = torch.stack([data[i:i + model.block_size] for i in ix])
        y = torch.stack([data[i + 1:i + model.block_size + 1] for i in ix])     # the same text, shifted by one
        _, loss = model(x, y)
        opt.zero_grad()
        loss.backward()
        opt.step()
        losses.append(loss.item())
    return losses

In [ ]:
# 检查 / checks
import codecs, contextlib, io, math
import torch

with contextlib.redirect_stdout(io.StringIO()):
    import this                                   # the Zen of Python, a small text that ships with Python
text = codecs.decode(this.s, "rot13")
chars = sorted(set(text))
stoi = {c: i for i, c in enumerate(chars)}
data = torch.tensor([stoi[c] for c in text])
V = len(chars)

torch.manual_seed(0)
model = MiniGPT(V)
x = data[:32][None]
logits, loss = model(x, data[1:33][None])
assert logits.shape == (1, 32, V) and loss.dim() == 0
assert model(x)[1] is None, "no targets, no loss"
assert abs(loss.item() - math.log(V)) < 0.6, f"an untrained model should start near ln(V) = {math.log(V):.2f}, got {loss.item():.2f}"

# causality: changing a later token must not change earlier logits
x2 = x.clone(); x2[0, 20:] = (x2[0, 20:] + 1) % V
assert torch.allclose(model(x)[0][0, :20], model(x2)[0][0, :20], atol=1e-5), "future tokens leak: check the mask"

losses = train(model, data, steps=300)
assert losses[-1] < 1.2, f"final loss {losses[-1]:.2f}: the model should learn this small text"
sample = model.generate(data[:8][None], 120, generator=torch.Generator().manual_seed(0))[0]
out = "".join(chars[i] for i in sample)
print(f"m25_minigpt: all tests passed (loss {losses[0]:.2f} -> {losses[-1]:.2f}, ln V = {math.log(V):.2f})")
print("  sample:", repr(out))

<details><summary><b>提示 1 / Hint 1</b></summary>

`self.attn(...)` 返回的是 `(输出, 权重)`，取第一个。

`self.attn(...)` returns `(output, weights)`. Take the first.

</details>

<details><summary><b>提示 2 / Hint 2</b></summary>

位置编号用 `torch.arange(T)`，因果掩码用 `torch.tril(torch.ones(T, T, dtype=torch.bool))`。

Positions are `torch.arange(T)` and the causal mask is `torch.tril(torch.ones(T, T, dtype=torch.bool))`.

</details>

<details><summary><b>提示 3 / Hint 3</b></summary>

计算损失时把 logits 变成 `(B*T, V)`、把 targets 变成 `(B*T,)`，再调用 `F.cross_entropy`。没有 targets 时损失返回 `None`。

For the loss reshape the logits to `(B*T, V)` and the targets to `(B*T,)` and call `F.cross_entropy`. Without targets return `None` as the loss.

</details>

<details><summary><b>参考答案 / Reference solution</b></summary>

```python
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


class Block(nn.Module):
    """A Pre-Norm Transformer block."""

    def __init__(self, d_model, n_head):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadAttention(d_model, n_head)
        self.ln2 = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(nn.Linear(d_model, 4 * d_model), nn.GELU(), nn.Linear(4 * d_model, d_model))

    def forward(self, x, mask):
        x = x + self.attn(self.ln1(x), mask)[0]
        x = x + self.mlp(self.ln2(x))
        return x


class MiniGPT(nn.Module):
    def __init__(self, vocab_size, block_size=32, d_model=64, n_head=4, n_layer=2):
        super().__init__()
        self.block_size = block_size
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(block_size, d_model)
        self.blocks = nn.ModuleList([Block(d_model, n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, idx, targets=None):
        """idx: (B, T) token ids. Returns (logits (B, T, V), loss or None)."""
        B, T = idx.shape
        x = self.tok_emb(idx) + self.pos_emb(torch.arange(T))
        mask = torch.tril(torch.ones(T, T, dtype=torch.bool))
        for block in self.blocks:
            x = block(x, mask)
        logits = self.head(self.ln_f(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(B * T, -1), targets.view(B * T))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, generator=None):
        for _ in range(max_new_tokens):
            logits, _ = self(idx[:, -self.block_size:])
            probs = torch.softmax(logits[:, -1], dim=-1)
            idx = torch.cat([idx, torch.multinomial(probs, 1, generator=generator)], dim=1)
        return idx


def train(model, data, steps=300, batch_size=32, lr=3e-3, seed=0):
    """Train on a 1-D tensor of token ids. Returns the list of loss values. Nothing to change here."""
    g = torch.Generator().manual_seed(seed)
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    losses = []
    for _ in range(steps):
        ix = torch.randint(len(data) - model.block_size - 1, (batch_size,), generator=g)
        x = torch.stack([data[i:i + model.block_size] for i in ix])
        y = torch.stack([data[i + 1:i + model.block_size + 1] for i in ix])     # the same text, shifted by one
        _, loss = model(x, y)
        opt.zero_grad()
        loss.backward()
        opt.step()
        losses.append(loss.item())
    return losses
```

</details>